# NetCore TAC Desk: Are the fixed ETAs accurate?

Load the numeric fields and flags with `np.genfromtxt`. Encode S1-S4 as integers 1-4 and preserve missing first-response times as `NaN`. Missing response times are still unknown, not zero-minute responses, and do not prevent us from comparing actual resolution hours with ETAs.

Run from the workspace root or the `notebooks` folder. This reader fits the synthetic export, whose text fields do not contain commas; `genfromtxt` does not handle quoted commas in general CSVs.

In [1]:
from pathlib import Path
from timeit import repeat

import numpy as np

workspace_root = Path.cwd()
if workspace_root.name == "notebooks":
    workspace_root = workspace_root.parent
csv_path = workspace_root / "data" / "tac_tickets.csv"

severity_labels = ("S1", "S2", "S3", "S4")


def parse_severity(value):
    return severity_labels.index(value) + 1


def parse_flag(value):
    return bool(("False", "True").index(value))


fields = [
    ("severity", "i4"), ("devices_affected", "i4"), ("software_version_months", "i4"),
    ("prior_ticket_90_days", "?"), ("engineer_experience_years", "f8"),
    ("first_response_minutes", "f8"), ("escalation_count", "i4"), ("reopened", "?"),
    ("sla_breached", "?"), ("resolution_hours", "f8"),
]
tickets = np.genfromtxt(
    csv_path, delimiter=",", names=True, usecols=[name for name, _ in fields], dtype=fields,
    converters={"severity": parse_severity, "prior_ticket_90_days": parse_flag,
                "reopened": parse_flag, "sla_breached": parse_flag},
    encoding="utf-8", comments=None, loose=False, ndmin=1,
)

severity = tickets["severity"]
actual_hours = tickets["resolution_hours"]
if tickets.size == 0 or not np.isfinite(actual_hours).all():
    raise ValueError("The ETA comparison needs tickets with finite resolution hours.")

missing_responses = np.count_nonzero(np.isnan(tickets["first_response_minutes"]))
print(f"Loaded {tickets.size:,} tickets; {missing_responses} have no recorded first-response time.")

Loaded 2,500 tickets; 50 have no recorded first-response time.


## Inspect the array

`ndim`, `shape`, and `size` describe the array. This structured array is one-dimensional because each element is a ticket with named fields. The field dtypes distinguish counts and codes, Boolean flags, and fractional hours.

In [2]:
print("Dimensions (ndim):", tickets.ndim)
print("Shape:", tickets.shape)
print("Size (tickets):", tickets.size)
print("Field dtypes:")
for name, _ in fields:
    print(f"  {name}: {tickets[name].dtype}")

Dimensions (ndim): 1
Shape: (2500,)
Size (tickets): 2500
Field dtypes:
  severity: int32
  devices_affected: int32
  software_version_months: int32
  prior_ticket_90_days: bool
  engineer_experience_years: float64
  first_response_minutes: float64
  escalation_count: int32
  reopened: bool
  sla_breached: bool
  resolution_hours: float64


## Index, slice, and filter tickets

Inspect one value, then preview the first five. Use Boolean masks to isolate S1 tickets for critical-incident review and tickets taking over 72 hours for long-running case review. The 72-hour filter is **not** the SLA breach rule: an S1 ticket can breach after just 4 hours.

In [3]:
print(f"First ticket's actual resolution time: {actual_hours[0]:.2f} h")
print("First five actual resolution times (h):", actual_hours[:5])

s1_mask = severity == 1
over_72_mask = actual_hours > 72
s1_tickets = tickets[s1_mask]
long_running_tickets = tickets[over_72_mask]
print(f"S1 tickets: {s1_tickets.size:,}")
print(f"Tickets taking more than 72 h: {long_running_tickets.size:,}")

First ticket's actual resolution time: 34.57 h
First five actual resolution times (h): [34.57 84.88 18.62 59.21 67.46]
S1 tickets: 161
Tickets taking more than 72 h: 287


## Apply the fixed ETA to every ticket

Fancy indexing looks up the ETA for all tickets at once. Subtract one from the severity code to get its zero-based position in the ETA table. Signed error is `actual - ETA`: positive means the customer waited longer than estimated; negative means the ticket was resolved sooner.

In [4]:
fixed_eta_hours = np.array([4.0, 24.0, 48.0, 72.0])
eta_hours = fixed_eta_hours[severity - 1]
error_hours = actual_hours - eta_hours

print("First five actual hours: ", actual_hours[:5])
print("First five fixed ETAs:    ", eta_hours[:5])
print("First five errors (hours):", np.round(error_hours[:5], 2))

First five actual hours:  [34.57 84.88 18.62 59.21 67.46]
First five fixed ETAs:     [24. 48. 24. 72. 72.]
First five errors (hours): [ 10.57  36.88  -5.38 -12.79  -4.54]


## Summarize resolution time and SLA breaches

MTTR is the mean resolution time; the median is the midpoint. The 25th and 75th percentiles bracket the middle half, and about 1 in 10 tickets takes longer than the 90th percentile. Compare MTTR by severity for staffing context. The SLA breach rate counts only tickets whose actual time **exceeds** their own severity's ETA.

In [5]:
mttr = np.mean(actual_hours)
median_hours = np.median(actual_hours)
p25, p75, p90 = np.percentile(actual_hours, [25, 75, 90])
breached = actual_hours > eta_hours
sla_breach_rate = np.mean(breached)

print(f"Overall MTTR: {mttr:.2f} h; median: {median_hours:.2f} h")
print(f"Resolution time percentiles: P25={p25:.2f} h, P75={p75:.2f} h, P90={p90:.2f} h")
for code, label in enumerate(severity_labels, start=1):
    group_hours = actual_hours[severity == code]
    if group_hours.size:
        print(f"{label} MTTR: {np.mean(group_hours):.2f} h ({group_hours.size:,} tickets)")
    else:
        print(f"{label} MTTR: not available (no tickets)")
print(f"SLA breach rate: {sla_breach_rate:.2%} ({np.count_nonzero(breached):,} tickets)")
print(f"Disagreements with stored SLA flags: {np.count_nonzero(breached != tickets['sla_breached'])}")

Overall MTTR: 39.26 h; median: 32.06 h
Resolution time percentiles: P25=19.55 h, P75=51.13 h, P90=75.90 h
S1 MTTR: 4.29 h (161 tickets)
S2 MTTR: 22.83 h (586 tickets)
S3 MTTR: 41.93 h (1,210 tickets)
S4 MTTR: 61.40 h (543 tickets)
SLA breach rate: 31.60% (790 tickets)
Disagreements with stored SLA flags: 0


## Measure baseline mean absolute error (MAE)

MAE is the average miss in hours, regardless of whether an ETA was too short or too long. This full-dataset number describes the current policy on these synthetic tickets. To show improvement, a future model must beat the fixed-ETA MAE recalculated on the **same held-out tickets** used to score the model.

In [6]:
baseline_mae_hours = np.mean(np.abs(error_hours))
print(f"Fixed ETAs missed actual resolution time by {baseline_mae_hours:.2f} h on average (MAE).")

Fixed ETAs missed actual resolution time by 18.68 h on average (MAE).


## Reshape and stack four triage features

`reshape(-1, 1)` turns the severity vector into a column; `column_stack` places the other features alongside it. Each row represents one ticket and each column is known at triage: severity code, devices affected, software age in months, and prior-ticket flag (0 or 1). Keep actual hours as the separate target and leave post-triage outcomes out of the features.

In [7]:
severity_column = severity.reshape(-1, 1)
prior_ticket_flag = tickets["prior_ticket_90_days"].astype(np.int32)
X = np.column_stack((
    severity_column, tickets["devices_affected"], tickets["software_version_months"], prior_ticket_flag,
))
y = actual_hours

print("Severity column shape:", severity_column.shape)
print("Feature matrix shape (tickets, features):", X.shape)
print("Target shape:", y.shape)
print("First five rows [severity, devices, software age, prior ticket]:")
print(X[:5])

Severity column shape: (2500, 1)
Feature matrix shape (tickets, features): (2500, 4)
Target shape: (2500,)
First five rows [severity, devices, software age, prior ticket]:
[[ 2  6 15  0]
 [ 3 11 34  0]
 [ 2  7 20  0]
 [ 4  1 15  0]
 [ 4  2 18  0]]


## Compare vectorized MAE with a Python loop

Both versions look up each ticket's ETA, compute its absolute error, and average the errors. `timeit` repeats this calculation on already-loaded data without timing file loading or list conversion. The ratio illustrates how fast TAC Desk could recalculate the baseline in this environment; it is not a guaranteed speedup.

In [8]:
severity_list = severity.tolist()
actual_list = actual_hours.tolist()
eta_table = fixed_eta_hours.tolist()


def numpy_mae():
    return np.mean(np.abs(actual_hours - fixed_eta_hours[severity - 1]))


def loop_mae():
    total_error = 0.0
    for code, actual in zip(severity_list, actual_list, strict=True):
        total_error += abs(actual - eta_table[code - 1])
    return total_error / len(actual_list)


np.testing.assert_allclose([numpy_mae(), loop_mae()], baseline_mae_hours)
runs_per_repeat = 100
numpy_seconds = min(repeat(numpy_mae, number=runs_per_repeat, repeat=5)) / runs_per_repeat
loop_seconds = min(repeat(loop_mae, number=runs_per_repeat, repeat=5)) / runs_per_repeat

print(f"Both methods agree: {baseline_mae_hours:.2f} h MAE.")
print(f"NumPy: {numpy_seconds * 1_000_000:.1f} us per call; Python loop: {loop_seconds * 1_000_000:.1f} us per call")
print(f"Loop time / NumPy time: {loop_seconds / numpy_seconds:.1f}x")

Both methods agree: 18.68 h MAE.
NumPy: 17.7 us per call; Python loop: 143.0 us per call
Loop time / NumPy time: 8.1x
